# Space Debris Tracker — Análise Exploratória
**Global Solution 2026.1 | FIAP**

Este notebook demonstra o pipeline completo: coleta de TLEs, conversão XYZ, modelo de risco e agente RAG.

In [ ]:
# ── Célula 1: Importações e carregamento de dados ──────────────────────────
import sys
import os

# Adiciona o diretório raiz ao path para importar os módulos src/ai
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.ai.tle_processor import process_all_objects
from src.ai.collision_model import train_model, get_top_conjunctions, generate_training_data
from src.ai.rag_agent import answer_question

# Carrega e processa todos os objetos TLE
print('Carregando objetos orbitais...')
objetos = process_all_objects()

# Converte para DataFrame para facilitar análise
df = pd.DataFrame(objetos)

print(f'\nTotal de objetos carregados: {len(df)}')
print(f'Colunas disponíveis: {list(df.columns)}')
df.head()

In [ ]:
# ── Célula 2: Visualização 2D das órbitas ─────────────────────────────────

# Classifica objetos por faixa orbital
def classificar_orbita(altitude_km):
    """Classifica a órbita com base na altitude."""
    if altitude_km < 2000:
        return 'LEO'
    elif altitude_km <= 35786:
        return 'MEO'
    else:
        return 'GEO'

df['orbita'] = df['altitude_km'].apply(classificar_orbita)

# Mapeamento de cores por faixa orbital
cores = {'LEO': 'green', 'MEO': 'orange', 'GEO': 'blue'}
df['cor'] = df['orbita'].map(cores)

fig, ax = plt.subplots(figsize=(10, 10))

# Plota os objetos no plano XY
for orbita, grupo in df.groupby('orbita'):
    ax.scatter(grupo['x'], grupo['y'],
               c=cores[orbita], label=orbita, alpha=0.6, s=15)

# Círculo representando a Terra (raio 6371 km)
terra = plt.Circle((0, 0), 6371, color='steelblue', alpha=0.3, label='Terra')
ax.add_patch(terra)

ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
ax.set_title('Distribuição Orbital dos Debris Rastreados')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Célula 3: Análise por altitude ────────────────────────────────────────

# Contagem por faixa orbital
contagem = df['orbita'].value_counts()
print('Distribuição por faixa orbital:')
print(contagem.to_string())

# Gráfico de pizza
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.pie(
    contagem.values,
    labels=contagem.index,
    colors=[cores[o] for o in contagem.index],
    autopct='%1.1f%%',
    startangle=90
)
ax1.set_title('Distribuição por Faixa Orbital')

# Tabela de estatísticas de altitude por faixa
stats = df.groupby('orbita')['altitude_km'].agg(['mean', 'min', 'max']).round(2)
stats.columns = ['Média (km)', 'Mínimo (km)', 'Máximo (km)']

ax2.axis('off')
tabela = ax2.table(
    cellText=stats.values,
    rowLabels=stats.index,
    colLabels=stats.columns,
    cellLoc='center',
    loc='center'
)
tabela.auto_set_font_size(False)
tabela.set_fontsize(11)
tabela.scale(1.2, 1.8)
ax2.set_title('Estatísticas de Altitude por Faixa Orbital')

plt.tight_layout()
plt.show()

print('\nEstatísticas detalhadas:')
print(stats)

In [ ]:
# ── Célula 4: Treinamento e validação do modelo ───────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Treina o modelo e obtém métricas
print('Treinando modelo de risco de colisão...')
modelo = train_model()

# Gera dados de teste para visualizações
X, y = generate_training_data(n_samples=1000)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
y_pred = modelo.predict(X_teste)

# Calcula métricas
acuracia = accuracy_score(y_teste, y_pred)
precisao = precision_score(y_teste, y_pred, average='weighted', zero_division=0)
recall   = recall_score(y_teste, y_pred, average='weighted', zero_division=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap da matriz de confusão
cm = confusion_matrix(y_teste, y_pred)
rotulos = ['Baixo', 'Médio', 'Alto']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=rotulos, yticklabels=rotulos, ax=ax1)
ax1.set_title('Matriz de Confusão')
ax1.set_ylabel('Real')
ax1.set_xlabel('Predito')

# Importância das features
importancias = modelo.feature_importances_
features = ['Distância (km)', 'Velocidade Relativa (km/s)']
ax2.barh(features, importancias, color=['steelblue', 'coral'])
ax2.set_title('Importância das Features')
ax2.set_xlabel('Importância')
for i, v in enumerate(importancias):
    ax2.text(v + 0.005, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

print(f'\nMétricas do Modelo:')
print(f'  Acurácia : {acuracia:.4f}')
print(f'  Precisão : {precisao:.4f}')
print(f'  Recall   : {recall:.4f}')

In [ ]:
# ── Célula 5: Teste do agente RAG ─────────────────────────────────────────

perguntas = [
    'O que é a Síndrome de Kessler?',
    'Quais foram os maiores eventos de colisão orbital da história?',
    'Como é calculado o risco de colisão entre dois objetos?'
]

print('=' * 70)
print('AGENTE RAG — Especialista em Debris Espaciais')
print('=' * 70)

for i, pergunta in enumerate(perguntas, 1):
    print(f'\nPergunta {i}: {pergunta}')
    print('-' * 50)
    resposta = answer_question(pergunta)
    print(f'Resposta: {resposta}')
    print()